# Coin Toss

Flip a quantum coin a number of times and get back the heads and tails counts.

## Setup

In [ ]:
import getpass
import os

from moth_atlas import Atlas

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY
ENGINE = "coin-toss-v1"

## Try it

Set the parameters, then run the cell.

In [ ]:
params = {"shots": 10, "mode": "emu"}

job = client.jobs.submit(ENGINE, params=params)
client.jobs.wait(job.job_id, on_update=lambda s: print(s.status))
result = client.jobs.download(job.job_id)
result

## Parameters

| name | type | default | what it does |
|---|---|---|---|
| `shots` | integer, above 0 | 10 | Number of coin tosses to run. More shots bring the heads/tails split closer to 50/50. |
| `mode` | `emu` or `qpu` | `emu` | `emu` uses the platform's simulator and returns in seconds; `qpu` submits to real IBM Quantum hardware and can queue for minutes. |
| `backend_name` | string | none | IBM Quantum backend to run on when mode is `qpu`. Leave blank to let the platform choose the least busy one. |

**Returns:** `heads` and `tails` (counts), `shots` (the number of measurements taken) and `output` (whichever had more counts).

## How it works

A single qubit is put into equal superposition with a Hadamard gate and measured `shots` times. The outcomes are counted and the majority is reported as the result.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Shots

More shots bring the heads/tails split closer to 50/50.

In [ ]:
for shots in (10, 100, 1000):
    job = client.jobs.run(ENGINE, params={**params, "shots": shots})
    r = client.jobs.download(job.job_id)
    print(f"{shots:>5} shots: {r['heads']} heads, {r['tails']} tails ({r['heads'] / shots:.0%} heads)")

## Questions

**Is this a random-number service?**

No. The circuit is one qubit; results are a probability sample, not a random-number service with guarantees.

**Why did my qpu run fail?**

A run still going after 300 s is cancelled and the job ends failed. Device queue times can be minutes to hours.